# Fire Watcher 🛰️🔥 — Treinamento no Google Colab (FLAME 3)

Este notebook executa o pipeline oficial de treinamento e avaliação comparativa dos dois modelos de visão computacional para segmentação binária de fogo (pixel a pixel):
1. **Modelo Óptico (RGB Corrigido):** U-Net com encoder ResNet34 pré-treinado no ImageNet.
2. **Modelo Térmico (Thermal Raw JPG):** U-Net com encoder ResNet34 pré-treinado na paleta térmica.

> **IMPORTANTE:** Certifique-se de ativar a GPU no Colab em: *Ambiente de Execução > Alterar tipo de ambiente de execução > Acelerador de hardware: T4 GPU*.

## 1. Montagem do Google Drive
Conecte sua conta do Google Drive para que os checkpoints dos modelos (.pth) e métricas sejam salvos permanentemente.

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    print("✅ Google Drive montado com sucesso!")
except Exception as e:
    print("⚠️ Não foi possível montar o Drive. Os arquivos serão salvos localmente na sessão do Colab.")

## 2. Clonagem do Repositório e Instalação de Dependências

In [ ]:
import os

# Clona o repositório se ainda não existir no ambiente efêmero
if not os.path.exists('/content/fire-watcher'):
    !git clone https://github.com/mikmathiaz/TCC_FireProject-API.git /content/fire-watcher

%cd /content/fire-watcher
!git pull

# Instalação das dependências oficiais de Machine Learning
!pip install -r ml/requirements-ml.txt -q

## 3. Verificação do Acelerador GPU (CUDA)

In [ ]:
import torch
print(f"Versão PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ ATENÇÃO: Nenhuma GPU detectada. Vá em 'Ambiente de Execução' > 'Alterar tipo de ambiente de execução' e selecione 'T4 GPU'.")

## 4. Localização e Descompactação do Pacote do Dataset (`flame3_colab_bundle.zip`)

Você pode:
- **Opção A (Recomendada):** Fazer upload do `flame3_colab_bundle.zip` para qualquer pasta do seu Google Drive.
- **Opção B:** Arrastar e soltar o `flame3_colab_bundle.zip` diretamente na aba de arquivos do Colab (à esquerda).

In [ ]:
import zipfile
import os
from pathlib import Path

DEST_DATA_DIR = "/content/fire-watcher/ml/data/flame3_bundle"

# Busca inteligente do arquivo zip em locais comuns
candidatos_zip = [
    "/content/flame3_colab_bundle.zip",
    "/content/drive/MyDrive/TCC_FireWatcher/flame3_colab_bundle.zip",
    "/content/drive/MyDrive/flame3_colab_bundle.zip",
]

# Se não encontrou nos caminhos fixos, procura recursivamente no Drive
caminho_zip = next((c for c in candidatos_zip if os.path.exists(c)), None)
if caminho_zip is None and os.path.exists('/content/drive/MyDrive'):
    achados = list(Path('/content/drive/MyDrive').rglob('flame3_colab_bundle.zip'))
    if achados:
        caminho_zip = str(achados[0])

if caminho_zip and os.path.exists(caminho_zip):
    print(f"📦 Encontrado bundle em: {caminho_zip}")
    print(f"Descompactando em {DEST_DATA_DIR}...")
    os.makedirs(DEST_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
        zip_ref.extractall(DEST_DATA_DIR)
    print("✅ Descompactação concluída! Dataset pronto para treino.")
else:
    print("❌ Arquivo 'flame3_colab_bundle.zip' não encontrado!")
    print("Faça upload do arquivo .zip no Google Drive ou arraste-o diretamente para o painel de arquivos do Colab.")

## 5. Modo de Sanidade / Smoke Test (--smoke)

Treina em apenas 8 amostras de fogo por 30 épocas para verificar capacidade de convergência rápida. O IoU deve superar 0.80 para comprovar que o pipeline está 100% íntegro.

In [ ]:
!python ml/scripts/train.py \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --smoke

## 6. Treinamento Modelo 1: RGB Corrigido (40 Épocas)

Os pesos salvos (.pth) e o histórico de métricas (.csv) são gravados diretamente no Google Drive para persistência definitiva.

In [ ]:
# Diretório persistente de checkpoints
if os.path.exists('/content/drive/MyDrive'):
    CHECKPOINT_DIR = "/content/drive/MyDrive/TCC_FireWatcher/checkpoints"
else:
    CHECKPOINT_DIR = "/content/fire-watcher/ml/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!python ml/scripts/train.py \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --output-dir {CHECKPOINT_DIR} \
    --epochs 40 \
    --batch-size 4 \
    --lr 1e-4

## 7. Treinamento Modelo 2: Térmico Raw JPG (40 Épocas)

In [ ]:
!python ml/scripts/train.py \
    --modality thermal \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --output-dir {CHECKPOINT_DIR} \
    --epochs 40 \
    --batch-size 4 \
    --lr 1e-4

## 8. Avaliação Acadêmica no Conjunto de Teste (Test Split)

Avalia ambos os modelos no bloco temporal final (dados futuros nunca vistos pelo modelo) e afere a latência de inferência em ms por imagem.

In [ ]:
print("=== AVALIAÇÃO MODELO RGB ===")
!python ml/scripts/evaluate.py \
    --checkpoint {CHECKPOINT_DIR}/best_model_rgb.pth \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --split test \
    --output-json {CHECKPOINT_DIR}/eval_results_rgb.json

print("\n" + "="*80 + "\n")

print("=== AVALIAÇÃO MODELO TÉRMICO ===")
!python ml/scripts/evaluate.py \
    --checkpoint {CHECKPOINT_DIR}/best_model_thermal.pth \
    --modality thermal \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --split test \
    --output-json {CHECKPOINT_DIR}/eval_results_thermal.json